# Chapter 17 – Recursive algorithms: sum, Merge Sort, binary search, maximum subarray (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms_java.ipynb)

The algorithms with pseudo-code of the chapter, run on the examples of the notes: the **sum of an array** (iterative, by halving and by decrease by one), **Merge Sort** with `Merge`, the **recursive binary search** and the **maximum subarray** problem (brute force, divide and conquer, Kadane's algorithm). The output reproduces the traces and the numbers of the chapter (with indices from 0).

The full chapter, with pseudo-code, proofs and worked examples, is [on the website](https://fabiofurini.github.io/algorithms/recursion/17-recursive-algorithms/).

Each program is written to a `.java` file by the `%%writefile` cell and then compiled and run with `javac` and `java`: run the cells in order, from top to bottom (the first cell checks or installs the JDK). No external libraries.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `recursiveSum` (camelCase).

In [1]:
# Check (and if needed install) the JDK
!java -version || apt-get -qq install -y default-jdk-headless

openjdk version "25.0.4.1" 2026-08-18
OpenJDK Runtime Environment (build 25.0.4.1+1-1-26.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 25.0.4.1+1-1-26.04.4-Ubuntu, mixed mode, sharing)


## 1. The sum of an array: SumIterative, RecursiveSum, RecursiveSumLinear

- **SumIterative**$(a,n)$: a loop that accumulates the partial sums. $S(n)=3n+4$, extra memory $\Theta(1)$.
- **RecursiveSum**$(a,l,r)$: divide and conquer on the two halves (base case $l=r$). $2n-1$ calls, $S(n)=7n-5$, depth $\lceil\log_2 n\rceil+1$.
- **RecursiveSumLinear**$(a,n)$: decrease by one: $S(n)=3n-1$, depth $n$.

*Input*: an array of $n\ge1$ integers (for `RecursiveSum`, two indices $l\le r$). *Output*: the sum of the elements. The counters `steps` (lines of pseudo-code executed), `calls` and `maxDepth` (simultaneously active calls) measure the quantities analysed in the chapter; in `RecursiveSum` and `RecursiveSumLinear` there is a single `return` so that the depth can be updated.

On the array of the example $a=[9,8,12,3,5,14,6]$ ($n=7$) the sum is $57$; `RecursiveSum` makes $13$ calls and $44$ steps, with depth $4$. Then we run the trace of the table of the chapter on $a=[9,8,12,3,5]$: the column *stack* reports the active calls.

In [2]:
%%writefile Sums.java
import java.util.*;

public class Sums {
    static long steps = 0;   // steps executed (lines of pseudo-code)
    static int calls = 0;    // calls executed
    static int depth = 0;    // calls active right now
    static int maxDepth = 0; // maximum depth of the recursion

    static void reset() {
        steps = 0;
        calls = 0;
        depth = 0;
        maxDepth = 0;
    }

    // Iterative sum of a[0..n-1]
    static int sumIterative(int[] a, int n) {
        int s = 0;
        int i = 0;
        steps += 2;                 // lines 1-2: s := 0, i := 1
        while (i < n) {
            steps += 3;             // true test + two lines of the body
            s = s + a[i];
            i = i + 1;
        }
        steps += 2;                 // false test + return
        return s;
    }

    // Sum of a[l..r] (0-based indices), l <= r, by divide and conquer
    static int recursiveSum(int[] a, int l, int r) {
        calls++;
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        steps++;                    // line 1: if l = r
        int result;
        if (l == r) {
            steps++;                // line 2: return a[l] (base case)
            result = a[l];
        } else {
            steps += 4;             // lines 3-6: m, two calls, return
            int m = l + (r - l) / 2;                // divide
            int s1 = recursiveSum(a, l, m);         // conquer (left half)
            int s2 = recursiveSum(a, m + 1, r);     // conquer (right half)
            result = s1 + s2;                       // combine
        }
        depth--;
        return result;
    }

    // Sum of a[0..n-1] by decrease by one, n >= 1
    static int recursiveSumLinear(int[] a, int n) {
        calls++;
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        steps++;                    // line 1: if n = 1
        int result;
        if (n == 1) {
            steps++;                // line 2: return a[1]
            result = a[0];
        } else {
            steps += 2;             // lines 3-4: call and return
            int s = recursiveSumLinear(a, n - 1);
            result = s + a[n - 1];
        }
        depth--;
        return result;
    }

    // ---- trace of RecursiveSum ----
    static int row = 0;
    static int maxStack = 0;

    static void emit(ArrayList<String> stack, String action) {
        row++;
        if (stack.size() > maxStack) maxStack = stack.size();
        System.out.printf("%2d | %s | %s%n", row, String.join(",", stack), action);
    }

    static String p(int l, int r) { return "(" + l + "," + r + ")"; }

    static int recursiveSumTrace(int[] a, int l, int r, ArrayList<String> stack) {
        stack.add(p(l, r));
        if (l == r) {
            emit(stack, "l=r => return a[" + l + "]=" + a[l]);
            stack.remove(stack.size() - 1);
            return a[l];
        }
        int m = l + (r - l) / 2;
        emit(stack, "l!=r, m=" + l + "+(" + r + "-" + l + ")/2=" + m + "; call " + p(l, m));
        int s1 = recursiveSumTrace(a, l, m, stack);
        emit(stack, "s1=" + s1 + "; call " + p(m + 1, r));
        int s2 = recursiveSumTrace(a, m + 1, r, stack);
        emit(stack, "s2=" + s2 + "; return " + s1 + "+" + s2 + "=" + (s1 + s2));
        stack.remove(stack.size() - 1);
        return s1 + s2;
    }

    public static void main(String[] args) {
        int[] a = {9, 8, 12, 3, 5, 14, 6};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        reset();
        int s = sumIterative(a, n);
        System.out.println("SumIterative       = " + s + "   steps=" + steps + " (3n+4=" + (3 * n + 4) + ")");
        reset();
        s = recursiveSum(a, 0, n - 1);
        System.out.println("RecursiveSum       = " + s + "   calls=" + calls + " (2n-1=" + (2 * n - 1) + ")  steps=" + steps + " (7n-5=" + (7 * n - 5) + ")  maxDepth=" + maxDepth);
        reset();
        s = recursiveSumLinear(a, n);
        System.out.println("RecursiveSumLinear = " + s + "   calls=" + calls + "  steps=" + steps + " (3n-1=" + (3 * n - 1) + ")  maxDepth=" + maxDepth);

        int[] b = {9, 8, 12, 3, 5};
        System.out.println();
        System.out.println("a = " + Arrays.toString(b) + "   RecursiveSum(a, 0, 4)");
        System.out.println(" # | stack | action");
        int res = recursiveSumTrace(b, 0, 4, new ArrayList<>());
        System.out.println("result " + res + ", rows " + row + ", maximum number of active calls " + maxStack);
    }
}

Writing Sums.java


In [3]:
!javac Sums.java && java Sums

a = [9, 8, 12, 3, 5, 14, 6]   n = 7
SumIterative       = 57   steps=25 (3n+4=25)
RecursiveSum       = 57   calls=13 (2n-1=13)  steps=44 (7n-5=44)  maxDepth=4
RecursiveSumLinear = 57   calls=7  steps=20 (3n-1=20)  maxDepth=7

a = [9, 8, 12, 3, 5]   RecursiveSum(a, 0, 4)
 # | stack | action
 1 | (0,4) | l!=r, m=0+(4-0)/2=2; call (0,2)
 2 | (0,4),(0,2) | l!=r, m=0+(2-0)/2=1; call (0,1)
 3 | (0,4),(0,2),(0,1) | l!=r, m=0+(1-0)/2=0; call (0,0)
 4 | (0,4),(0,2),(0,1),(0,0) | l=r => return a[0]=9
 5 | (0,4),(0,2),(0,1) | s1=9; call (1,1)
 6 | (0,4),(0,2),(0,1),(1,1) | l=r => return a[1]=8
 7 | (0,4),(0,2),(0,1) | s2=8; return 9+8=17
 8 | (0,4),(0,2) | s1=17; call (2,2)
 9 | (0,4),(0,2),(2,2) | l=r => return a[2]=12
10 | (0,4),(0,2) | s2=12; return 17+12=29
11 | (0,4) | s1=29; call (3,4)
12 | (0,4),(3,4) | l!=r, m=3+(4-3)/2=3; call (3,3)
13 | (0,4),(3,4),(3,3) | l=r => return a[3]=3
14 | (0,4),(3,4) | s1=3; call (4,4)
15 | (0,4),(3,4),(4,4) | l=r => return a[4]=5
16 | (0,4),(3,4) | s2=5; retur

**Try it yourself.** Change the array `a` (also its length) and compare steps, calls and depth: only `RecursiveSumLinear` has depth $n$.

In [4]:
%%writefile ProvaSums.java
import java.util.*;

public class ProvaSums {
    public static void main(String[] args) {
        // Change the array: compare steps, calls and depth of the three algorithms.
        int[] a = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16};
        int n = a.length;
        Sums.reset(); int s = Sums.sumIterative(a, n);
        System.out.println(s + " " + Sums.steps);
        Sums.reset(); s = Sums.recursiveSum(a, 0, n - 1);
        System.out.println(s + " " + Sums.steps + " " + Sums.calls + " " + Sums.maxDepth);
        Sums.reset(); s = Sums.recursiveSumLinear(a, n);
        System.out.println(s + " " + Sums.steps + " " + Sums.calls + " " + Sums.maxDepth);
    }
}

Writing ProvaSums.java


In [5]:
!javac ProvaSums.java && java ProvaSums

136 52
136 107 31 5
136 47 16 16


## 2. Merge Sort and Merge

- **MergeSort**$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the sub-array $a[l..r]$ sorted in non-decreasing order. It is called `MergeSort(a, 1, n)` in the pseudo-code, `merge_sort(a, 0, n-1)` in the code.
- **Merge**$(a,l,m,r)$ — *Input*: an array and indices $l\le m<r$ such that $a[l..m]$ and $a[m+1..r]$ are sorted. *Output*: $a[l..r]$ sorted. It copies the two halves into $L$ and $R$ and moves back into $a$, step by step, the smaller of the first two elements not yet copied.

The counter `comparisons` counts the comparisons between elements executed by `Merge`. On the array of the example $a=[9,12,3,16,19,11,12]$ we print the calls of `Merge` in execution order (with the array after each of them, table of the chapter; indices are from 0: adding 1 gives those of the chapter), then the trace of `Merge(a, 0, 1, 3)` on $a=[5,8,2,7]$ (example of the C++ implementation).

In [6]:
%%writefile MergeSort.java
import java.util.*;

public class MergeSort {
    static long comparisons = 0;   // comparisons between elements executed by Merge
    static int msCalls = 0;        // calls of MergeSort

    // merge(a, l, m, r): merges the sorted a[l..m] and a[m+1..r]
    static void merge(int[] a, int l, int m, int r) {
        int n1 = m - l + 1;             // size of the left part
        int n2 = r - m;                 // size of the right part
        int[] L = new int[n1];          // auxiliary arrays
        int[] R = new int[n2];
        for (int i = 0; i < n1; i++) {
            L[i] = a[l + i];
        }
        for (int j = 0; j < n2; j++) {
            R[j] = a[m + 1 + j];
        }
        int i = 0, j = 0, k = l;
        while (i < n1 && j < n2) {      // merge L and R into a[l..r]
            comparisons++;
            if (L[i] <= R[j]) {
                a[k] = L[i];
                i++;
            } else {
                a[k] = R[j];
                j++;
            }
            k++;
        }
        while (i < n1) {                // copy the rest of L
            a[k] = L[i];
            i++;
            k++;
        }
        while (j < n2) {                // copy the rest of R
            a[k] = R[j];
            j++;
            k++;
        }
    }

    // mergeSort(a, l, r): sorts a[l..r]
    static void mergeSort(int[] a, int l, int r) {
        msCalls++;
        if (l < r) {
            int m = l + (r - l) / 2;    // divide
            mergeSort(a, l, m);         // conquer (left half)
            mergeSort(a, m + 1, r);     // conquer (right half)
            merge(a, l, m, r);          // combine
        }
    }

    // ---- variants that print the trace ----
    static void mergeSortLog(int[] a, int l, int r) {
        if (l < r) {
            int m = l + (r - l) / 2;
            mergeSortLog(a, l, m);
            mergeSortLog(a, m + 1, r);
            String left = Arrays.toString(Arrays.copyOfRange(a, l, m + 1));
            String right = Arrays.toString(Arrays.copyOfRange(a, m + 1, r + 1));
            merge(a, l, m, r);
            System.out.println("Merge(a," + l + "," + m + "," + r + ")  " + left + " and " + right + "  ->  " + Arrays.toString(a));
        }
    }

    static void mergeTrace(int[] a, int l, int m, int r) {
        int n1 = m - l + 1, n2 = r - m;
        int[] L = Arrays.copyOfRange(a, l, m + 1), R = Arrays.copyOfRange(a, m + 1, r + 1);
        System.out.println("Merge(a, " + l + ", " + m + ", " + r + ") on a = " + Arrays.toString(a));
        System.out.println("n1=" + n1 + ", n2=" + n2 + ", L=" + Arrays.toString(L) + ", R=" + Arrays.toString(R));
        int i = 0, j = 0, k = l, it = 0;
        while (i < n1 && j < n2) {
            it++;
            String head;
            if (L[i] <= R[j]) {
                head = String.format("%2d. loop 1: L[%d]=%d <= R[%d]=%d? Yes => a[%d] = L[%d] = %d", it, i, L[i], j, R[j], k, i, L[i]);
                a[k] = L[i];
                i++;
            } else {
                head = String.format("%2d. loop 1: L[%d]=%d <= R[%d]=%d? No => a[%d] = R[%d] = %d", it, i, L[i], j, R[j], k, j, R[j]);
                a[k] = R[j];
                j++;
            }
            k++;
            System.out.println(head + ", i=" + i + ", j=" + j + ", k=" + k + "; a=" + Arrays.toString(a));
        }
        while (i < n1) {
            a[k] = L[i];
            System.out.println("    loop 2 (rest of L): a[" + k + "] = L[" + i + "] = " + L[i] + "; a=" + Arrays.toString(a));
            i++;
            k++;
        }
        while (j < n2) {
            a[k] = R[j];
            System.out.println("    loop 3 (rest of R): a[" + k + "] = R[" + j + "] = " + R[j] + "; a=" + Arrays.toString(a));
            j++;
            k++;
        }
        System.out.println("result a = " + Arrays.toString(a) + ", comparisons = " + it);
    }

    public static void main(String[] args) {
        int[] a = {9, 12, 3, 16, 19, 11, 12};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        System.out.println("Calls of Merge in execution order:");
        mergeSortLog(a.clone(), 0, n - 1);
        comparisons = 0;
        msCalls = 0;
        mergeSort(a, 0, n - 1);
        System.out.println("sorted = " + Arrays.toString(a) + "   calls of MergeSort=" + msCalls + " (2n-1=" + (2 * n - 1) + ")   comparisons=" + comparisons);
        System.out.println();
        int[] b = {5, 8, 2, 7};
        mergeTrace(b, 0, 1, 3);
    }
}

Writing MergeSort.java


In [7]:
!javac MergeSort.java && java MergeSort

a = [9, 12, 3, 16, 19, 11, 12]   n = 7
Calls of Merge in execution order:
Merge(a,0,0,1)  [9] and [12]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,2,2,3)  [3] and [16]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,0,1,3)  [9, 12] and [3, 16]  ->  [3, 9, 12, 16, 19, 11, 12]
Merge(a,4,4,5)  [19] and [11]  ->  [3, 9, 12, 16, 11, 19, 12]
Merge(a,4,5,6)  [11, 19] and [12]  ->  [3, 9, 12, 16, 11, 12, 19]
Merge(a,0,3,6)  [3, 9, 12, 16] and [11, 12, 19]  ->  [3, 9, 11, 12, 12, 16, 19]
sorted = [3, 9, 11, 12, 12, 16, 19]   calls of MergeSort=13 (2n-1=13)   comparisons=14

Merge(a, 0, 1, 3) on a = [5, 8, 2, 7]
n1=2, n2=2, L=[5, 8], R=[2, 7]
 1. loop 1: L[0]=5 <= R[0]=2? No => a[0] = R[0] = 2, i=0, j=1, k=1; a=[2, 8, 2, 7]
 2. loop 1: L[0]=5 <= R[1]=7? Yes => a[1] = L[0] = 5, i=1, j=1, k=2; a=[2, 5, 2, 7]
 3. loop 1: L[1]=8 <= R[1]=7? No => a[2] = R[1] = 7, i=1, j=2, k=3; a=[2, 5, 7, 7]
    loop 2 (rest of L): a[3] = L[1] = 8; a=[2, 5, 7, 8]
result a = [2, 5, 7, 8], comparisons = 3


**Try it yourself.** Sort another array: the number of comparisons of `Merge` is at most $n\lceil\log_2 n\rceil$ and the calls of `MergeSort` are always $2n-1$.

In [8]:
%%writefile ProvaMergeSort.java
import java.util.*;

public class ProvaMergeSort {
    public static void main(String[] args) {
        // Sort another array and count the comparisons (at most n*ceil(log2 n)).
        int[] b = {38, 27, 43, 3, 9, 82, 10, 5};
        MergeSort.comparisons = 0;
        MergeSort.msCalls = 0;
        MergeSort.mergeSort(b, 0, b.length - 1);
        System.out.println(Arrays.toString(b) + " " + MergeSort.comparisons + " " + MergeSort.msCalls);
    }
}

Writing ProvaMergeSort.java


In [9]:
!javac ProvaMergeSort.java && java ProvaMergeSort

[3, 5, 9, 10, 27, 38, 43, 82] 17 15


## 3. RecursiveBinarySearch

**RecursiveBinarySearch**$(a,left,right,m)$ — *Input*: an array $a$ of natural numbers sorted in non-decreasing order, two indices $left$ and $right$ and a value $m$. *Output*: an index $i\in\{left,\dots,right\}$ with $a[i]=m$, or $-1$ if $m$ is not in $a[left..right]$. Every call makes **at most one** recursive call on (about) half of the interval: the call tree is a path, with at most $\lfloor\log_2 n\rfloor+2$ calls. To search the whole array we call `RecursiveBinarySearch(a, 0, n-1, m)`.

On the same array $a=[3,9,11,12,12,16,19]$ of the chapter we search $m=12,10,3,19$. For each search the table has one row per call ($left$, $right$, $mid$, $a[mid]$, action), as in the chapter but with indices from 0: adding 1 gives those of the chapter (for example $m=19$ is found at index 6, that is position 7).

In [10]:
%%writefile BinarySearch.java
import java.util.*;

public class BinarySearch {
    static int rbsCalls = 0;   // calls executed

    // Index of m in a[left..right] (sorted array), or -1
    static int recursiveBinarySearch(int[] a, int left, int right, int m) {
        rbsCalls++;
        if (left > right) {
            return -1;                  // empty interval: not found
        }
        int mid = left + (right - left) / 2;
        if (a[mid] == m) {
            return mid;                 // found
        }
        if (a[mid] < m) {
            return recursiveBinarySearch(a, mid + 1, right, m);    // right half
        }
        return recursiveBinarySearch(a, left, mid - 1, m);         // left half
    }

    // As above, but prints one row per call (k = number of the call)
    static int rbsTrace(int[] a, int left, int right, int m, int k) {
        if (left > right) {
            System.out.printf("%4d %4d %5d %3s %6s left > right => not found, return -1%n", k, left, right, "-", "-");
            return -1;
        }
        int mid = left + (right - left) / 2;
        System.out.printf("%4d %4d %5d %3d %6d ", k, left, right, mid, a[mid]);
        if (a[mid] == m) {
            System.out.println("a[mid] = m => found, return " + mid);
            return mid;
        }
        if (a[mid] < m) {
            System.out.println("a[mid] < m => call on left=" + (mid + 1) + ", right=" + right);
            return rbsTrace(a, mid + 1, right, m, k + 1);
        }
        System.out.println("a[mid] > m => call on left=" + left + ", right=" + (mid - 1));
        return rbsTrace(a, left, mid - 1, m, k + 1);
    }

    public static void main(String[] args) {
        int[] a = {3, 9, 11, 12, 12, 16, 19};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        int[] ms = {12, 10, 3, 19};
        for (int m : ms) {
            System.out.println();
            System.out.println("Search for m = " + m);
            System.out.printf("%4s %4s %5s %3s %6s action%n", "call", "left", "right", "mid", "a[mid]");
            rbsTrace(a, 0, n - 1, m, 1);
        }
        System.out.println();
        for (int m : ms) {
            rbsCalls = 0;
            int r = recursiveBinarySearch(a, 0, n - 1, m);
            System.out.printf("m=%-3d -> %2d (%2d with indices from 1)  calls=%d%n", m, r, r >= 0 ? r + 1 : -1, rbsCalls);
        }
    }
}

Writing BinarySearch.java


In [11]:
!javac BinarySearch.java && java BinarySearch

a = [3, 9, 11, 12, 12, 16, 19]   n = 7

Search for m = 12
call left right mid a[mid] action
   1    0     6   3     12 a[mid] = m => found, return 3

Search for m = 10
call left right mid a[mid] action
   1    0     6   3     12 a[mid] > m => call on left=0, right=2
   2    0     2   1      9 a[mid] < m => call on left=2, right=2
   3    2     2   2     11 a[mid] > m => call on left=2, right=1
   4    2     1   -      - left > right => not found, return -1

Search for m = 3
call left right mid a[mid] action
   1    0     6   3     12 a[mid] > m => call on left=0, right=2
   2    0     2   1      9 a[mid] > m => call on left=0, right=0
   3    0     0   0      3 a[mid] = m => found, return 0

Search for m = 19
call left right mid a[mid] action
   1    0     6   3     12 a[mid] < m => call on left=4, right=6
   2    4     6   5     16 a[mid] < m => call on left=6, right=6
   3    6     6   6     19 a[mid] = m => found, return 6

m=12  ->  3 ( 4 with indices from 1)  calls=1
m=10  -> -1 (

**Try it yourself.** Search another value (present or absent) and check that the calls are at most $\lfloor\log_2 n\rfloor+2$.

In [12]:
%%writefile ProvaBinarySearch.java
import java.util.*;

public class ProvaBinarySearch {
    public static void main(String[] args) {
        // Search another value, also an absent one: the calls are at most floor(log2 n) + 2.
        int[] a = {3, 9, 11, 12, 12, 16, 19};
        BinarySearch.rbsCalls = 0;
        System.out.println(BinarySearch.recursiveBinarySearch(a, 0, a.length - 1, 16) + " " + BinarySearch.rbsCalls);
        BinarySearch.rbsCalls = 0;
        System.out.println(BinarySearch.recursiveBinarySearch(a, 0, a.length - 1, 100) + " " + BinarySearch.rbsCalls);
    }
}

Writing ProvaBinarySearch.java


In [13]:
!javac ProvaBinarySearch.java && java ProvaBinarySearch

5 2
-1 4


## 4. BruteForceMaximumSubarray

**BruteForceMaximumSubarray**$(a,n)$ — *Input*: an array $a[1..n]$ of $n$ integers. *Output*: the indices $bl\le br$ of a maximum-sum subarray and the sum $best$. It examines all the $n(n+1)/2$ subarrays $a[l..r]$ (non-empty, of consecutive elements), computing the sums incrementally: $\Theta(n^2)$.

On the example $a=[-2,5,-1,3]$ of the chapter the table has one row for each subarray examined ($l$, $r$, sum, action; every row that does not improve shows the current $best$). Indices are from 0: the chapter finds $(2,4,7)$, that is $a[2..4]=[5,-1,3]$ with sum $7$, here $(1,3,7)$.

In [14]:
%%writefile MaxSubBrute.java
import java.util.*;

public class MaxSubBrute {
    // Returns {bl, br, best}: a maximum-sum subarray of a[0..n-1]
    static int[] bruteForceMaximumSubarray(int[] a, int n) {
        int best = a[0];
        int bl = 0;
        int br = 0;
        int l = 0;
        while (l < n) {
            int sum = 0;
            int r = l;
            while (r < n) {
                sum = sum + a[r];             // sum of a[l..r]
                if (sum > best) {
                    best = sum;
                    bl = l;
                    br = r;
                }
                r = r + 1;
            }
            l = l + 1;
        }
        return new int[]{bl, br, best};
    }

    // As above, but prints one row for each subarray examined
    static int[] bruteForceTrace(int[] a, int n) {
        int best = a[0];
        int bl = 0;
        int br = 0;
        for (int l = 0; l < n; l++) {
            int sum = 0;
            for (int r = l; r < n; r++) {
                sum = sum + a[r];
                if (sum > best) {
                    best = sum;
                    bl = l;
                    br = r;
                    System.out.printf("%3d %3d %5d  sum > best => best=%d, bl=%d, br=%d%n", l, r, sum, best, bl, br);
                } else {
                    System.out.printf("%3d %3d %5d  sum <= best=%d%n", l, r, sum, best);
                }
            }
        }
        return new int[]{bl, br, best};
    }

    public static void main(String[] args) {
        int[] a = {-2, 5, -1, 3};
        int n = 4;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n + "   subarrays n(n+1)/2 = " + n * (n + 1) / 2);
        System.out.printf("%3s %3s %5s  action%n", "l", "r", "sum");
        int[] t = bruteForceTrace(a, n);
        System.out.printf("result (bl, br, best) = (%d, %d, %d)   (%d, %d, %d with indices from 1)%n", t[0], t[1], t[2], t[0] + 1, t[1] + 1, t[2]);
        int[] u = bruteForceMaximumSubarray(a, n);
        System.out.printf("BruteForceMaximumSubarray(a, n) = (%d, %d, %d)%n", u[0], u[1], u[2]);
    }
}

Writing MaxSubBrute.java


In [15]:
!javac MaxSubBrute.java && java MaxSubBrute

a = [-2, 5, -1, 3]   n = 4   subarrays n(n+1)/2 = 10
  l   r   sum  action
  0   0    -2  sum <= best=-2
  0   1     3  sum > best => best=3, bl=0, br=1
  0   2     2  sum <= best=3
  0   3     5  sum > best => best=5, bl=0, br=3
  1   1     5  sum <= best=5
  1   2     4  sum <= best=5
  1   3     7  sum > best => best=7, bl=1, br=3
  2   2    -1  sum <= best=7
  2   3     2  sum <= best=7
  3   3     3  sum <= best=7
result (bl, br, best) = (1, 3, 7)   (2, 4, 7 with indices from 1)
BruteForceMaximumSubarray(a, n) = (1, 3, 7)


**Try it yourself.** Change the array (for example with only negative numbers): the maximum subarray is the single largest element.

In [16]:
%%writefile ProvaMaxSubBrute.java
import java.util.*;

public class ProvaMaxSubBrute {
    public static void main(String[] args) {
        // Try an array of only negative numbers: the maximum subarray is the single largest element.
        int[] b = {-3, -1, -2, -5};
        int[] t = MaxSubBrute.bruteForceMaximumSubarray(b, b.length);
        System.out.println("(" + t[0] + ", " + t[1] + ", " + t[2] + ")");
    }
}

Writing ProvaMaxSubBrute.java


In [17]:
!javac ProvaMaxSubBrute.java && java ProvaMaxSubBrute

(1, 1, -1)


## 5. FindMaxCrossingSubarray and FindMaximumSubarray (divide and conquer)

- **FindMaxCrossingSubarray**$(a,l,mid,r)$ — *Input*: an array and three indices $l\le mid<r$. *Output*: the indices $maxleft\le mid<maxright$ of a maximum-sum subarray among those crossing the midpoint, and its sum. It scans the left half from $mid$ down to $l$ and the right half from $mid+1$ to $r$: $\Theta(n)$.
- **FindMaximumSubarray**$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the indices $i\le j$ of a maximum-sum subarray of $a[l..r]$ and its sum. The best among the maximum subarray of the left half, of the right half and of those crossing the midpoint (in case of ties, the left one first, then the right one). $S(n)=2S(n/2)+\Theta(n)=\Theta(n\log_2 n)$, $2n-1$ calls.

The $-\infty$ of the pseudo-code is the smallest representable value. First the trace on the example $a=[-2,5,-1,3]$ (like the execution of the C++ implementation of the chapter; result $(1,3,7)$, that is $(2,4,7)$ with indices from 1), then the stock example, with the daily changes $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$: table of the results of the calls on the sub-intervals (by level of the tree, as in the chapter; indices are from 0) and the computations of the two `FindMaxCrossingSubarray` of level 1. The result is $(7,10,43)$, that is $(8,11,43)$ with indices from 1.

In [18]:
%%writefile MaxSubDC.java
import java.util.*;

public class MaxSubDC {
    static int fmsCalls = 0;   // calls of FindMaximumSubarray

    static String tr(int[] t) {
        return "(" + t[0] + ", " + t[1] + ", " + t[2] + ")";
    }

    // Returns {maxleft, maxright, sum} of the best subarray crossing mid
    static int[] findMaxCrossingSubarray(int[] a, int l, int mid, int r) {
        int leftsum = Integer.MIN_VALUE;        // -infinity
        int sum = 0;
        int maxleft = mid;
        for (int i = mid; i >= l; i--) {        // from mid down to l
            sum = sum + a[i];                   // sum of a[i..mid]
            if (sum > leftsum) {
                leftsum = sum;
                maxleft = i;
            }
        }
        int rightsum = Integer.MIN_VALUE;
        sum = 0;
        int maxright = mid + 1;
        for (int j = mid + 1; j <= r; j++) {    // from mid+1 to r
            sum = sum + a[j];                   // sum of a[mid+1..j]
            if (sum > rightsum) {
                rightsum = sum;
                maxright = j;
            }
        }
        return new int[]{maxleft, maxright, leftsum + rightsum};
    }

    // Returns {i, j, sum} of a maximum-sum subarray of a[l..r]
    static int[] findMaximumSubarray(int[] a, int l, int r) {
        fmsCalls++;
        if (l == r) {
            return new int[]{l, r, a[l]};       // base case: one element
        }
        int mid = l + (r - l) / 2;
        int[] left = findMaximumSubarray(a, l, mid);
        int[] right = findMaximumSubarray(a, mid + 1, r);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        if (left[2] >= right[2] && left[2] >= cross[2]) {
            return left;
        }
        if (right[2] >= left[2] && right[2] >= cross[2]) {
            return right;
        }
        return cross;
    }

    // ---- variants that print the trace ----
    static int[] best(int[] left, int[] right, int[] cross) {
        if (left[2] >= right[2] && left[2] >= cross[2]) return left;
        if (right[2] >= left[2] && right[2] >= cross[2]) return right;
        return cross;
    }

    static int[] fmsTrace(int[] a, int l, int r, int depth) {
        String pad = " ".repeat(2 * depth);
        if (l == r) {
            System.out.printf("%sFindMaximumSubarray(a, %d, %d) = (%d, %d, %d)  base case%n", pad, l, r, l, r, a[l]);
            return new int[]{l, r, a[l]};
        }
        int mid = l + (r - l) / 2;
        System.out.printf("%sFindMaximumSubarray(a, %d, %d): l != r, mid = %d%n", pad, l, r, mid);
        int[] left = fmsTrace(a, l, mid, depth + 1);
        int[] right = fmsTrace(a, mid + 1, r, depth + 1);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        System.out.printf("%s  FindMaxCrossingSubarray(a, %d, %d, %d) = %s%n", pad, l, mid, r, tr(cross));
        int[] ret = best(left, right, cross);
        System.out.printf("%s  left.sum=%d, right.sum=%d, cross.sum=%d => returns %s%n", pad, left[2], right[2], cross[2], tr(ret));
        return ret;
    }

    // each row: {l, r, mid} + left + right + cross + ret
    static int[] fmsRows(int[] a, int l, int r, ArrayList<int[][]> rows) {
        if (l == r) {
            return new int[]{l, r, a[l]};
        }
        int mid = l + (r - l) / 2;
        int[] left = fmsRows(a, l, mid, rows);
        int[] right = fmsRows(a, mid + 1, r, rows);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        int[] ret = best(left, right, cross);
        rows.add(new int[][]{{l, r, mid}, left, right, cross, ret});
        return ret;
    }

    static void crossingTrace(int[] a, int l, int mid, int r) {
        System.out.printf("FindMaxCrossingSubarray(a, %d, %d, %d)%n", l, mid, r);
        System.out.printf("%3s %5s %5s %8s %8s%n", "i", "a[i]", "sum", "leftsum", "maxleft");
        int leftsum = Integer.MIN_VALUE, maxleft = mid, sum = 0;
        for (int i = mid; i >= l; i--) {
            sum = sum + a[i];
            if (sum > leftsum) {
                leftsum = sum;
                maxleft = i;
            }
            System.out.printf("%3d %5d %5d %8d %8d%n", i, a[i], sum, leftsum, maxleft);
        }
        System.out.printf("%3s %5s %5s %8s %8s%n", "j", "a[j]", "sum", "rightsum", "maxright");
        int rightsum = Integer.MIN_VALUE, maxright = mid + 1;
        sum = 0;
        for (int j = mid + 1; j <= r; j++) {
            sum = sum + a[j];
            if (sum > rightsum) {
                rightsum = sum;
                maxright = j;
            }
            System.out.printf("%3d %5d %5d %8d %8d%n", j, a[j], sum, rightsum, maxright);
        }
        System.out.printf("crossing subarray a[%d..%d], sum %d + %d = %d%n", maxleft, maxright, leftsum, rightsum, leftsum + rightsum);
    }

    public static void main(String[] args) {
        int[] a = {-2, 5, -1, 3};
        System.out.println("a = " + Arrays.toString(a));
        int[] res = fmsTrace(a, 0, 3, 0);
        fmsCalls = 0;
        findMaximumSubarray(a, 0, 3);
        System.out.printf("result %s   (%d, %d, %d with indices from 1)   calls=%d (2n-1=%d)%n", tr(res), res[0] + 1, res[1] + 1, res[2], fmsCalls, 2 * 4 - 1);

        System.out.println();
        int[] b = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
        int n = 16;
        System.out.println("a = " + Arrays.toString(b));
        fmsCalls = 0;
        res = findMaximumSubarray(b, 0, n - 1);
        System.out.printf("FindMaximumSubarray(a, 0, %d) = %s   (%d, %d, %d with indices from 1)   calls=%d (2n-1=%d)%n", n - 1, tr(res), res[0] + 1, res[1] + 1, res[2], fmsCalls, 2 * n - 1);
        ArrayList<int[][]> rows = new ArrayList<>();
        fmsRows(b, 0, n - 1, rows);
        rows.sort((x, y) -> (x[0][1] - x[0][0]) != (y[0][1] - y[0][0]) ? (y[0][1] - y[0][0]) - (x[0][1] - x[0][0]) : x[0][0] - y[0][0]);
        System.out.printf("%-9s %3s  %-13s %-13s %-13s returned%n", "call", "mid", "left", "right", "crossing");
        for (int[][] w : rows) {
            String call = "[" + w[0][0] + ".." + w[0][1] + "]";
            System.out.printf("%-9s %3d  %-13s %-13s %-13s %s%n", call, w[0][2], tr(w[1]), tr(w[2]), tr(w[3]), tr(w[4]));
        }
        System.out.println();
        crossingTrace(b, 0, 3, 7);
        System.out.println();
        crossingTrace(b, 8, 11, 15);
    }
}

Writing MaxSubDC.java


In [19]:
!javac MaxSubDC.java && java MaxSubDC

a = [-2, 5, -1, 3]
FindMaximumSubarray(a, 0, 3): l != r, mid = 1
  FindMaximumSubarray(a, 0, 1): l != r, mid = 0
    FindMaximumSubarray(a, 0, 0) = (0, 0, -2)  base case
    FindMaximumSubarray(a, 1, 1) = (1, 1, 5)  base case
    FindMaxCrossingSubarray(a, 0, 0, 1) = (0, 1, 3)
    left.sum=-2, right.sum=5, cross.sum=3 => returns (1, 1, 5)
  FindMaximumSubarray(a, 2, 3): l != r, mid = 2
    FindMaximumSubarray(a, 2, 2) = (2, 2, -1)  base case
    FindMaximumSubarray(a, 3, 3) = (3, 3, 3)  base case
    FindMaxCrossingSubarray(a, 2, 2, 3) = (2, 3, 2)
    left.sum=-1, right.sum=3, cross.sum=2 => returns (3, 3, 3)
  FindMaxCrossingSubarray(a, 0, 1, 3) = (1, 3, 7)
  left.sum=5, right.sum=3, cross.sum=7 => returns (1, 3, 7)
result (1, 3, 7)   (2, 4, 7 with indices from 1)   calls=7 (2n-1=7)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
FindMaximumSubarray(a, 0, 15) = (7, 10, 43)   (8, 11, 43 with indices from 1)   calls=31 (2n-1=31)
call      mid  left          righ

**Try it yourself.** Change the array: the maximum subarray may have a single element or cross the midpoint; the calls are always $2n-1$.

In [20]:
%%writefile ProvaMaxSubDC.java
import java.util.*;

public class ProvaMaxSubDC {
    public static void main(String[] args) {
        // Change the array: how many elements does the maximum subarray have? The calls are always 2n-1.
        int[] b = {4, -1, 2, 1, -5, 4, -7, 3, 3};
        MaxSubDC.fmsCalls = 0;
        int[] t = MaxSubDC.findMaximumSubarray(b, 0, b.length - 1);
        System.out.println(MaxSubDC.tr(t) + " " + MaxSubDC.fmsCalls);
    }
}

Writing ProvaMaxSubDC.java


In [21]:
!javac ProvaMaxSubDC.java && java ProvaMaxSubDC

(0, 3, 6) 17


## 6. Kadane

**Kadane**$(a,n)$ — *Input*: an array $a[1..n]$ of $n$ integers. *Output*: the indices $bl\le br$ of a maximum-sum subarray and the sum $best$. A single scan: $cur$ is the maximum sum of a subarray **ending** at the current position and $cl$ its first index; if $cur<0$ we restart from $a[i]$, otherwise we extend. $\Theta(n)$ steps, $\Theta(1)$ extra memory. (Chapter 18 has the compact version that returns only the sum.)

On the stock example (daily changes $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$) we print the table of the chapter: one row per iteration, with the values at the end of the iteration (indices from 0, row $i=0$ is the initialization). The result $(7,10,43)$, that is $(8,11,43)$ with indices from 1, coincides with the one of the divide and conquer algorithm.

In [22]:
%%writefile Kadane.java
import java.util.*;

public class Kadane {
    // Returns {bl, br, best}: a maximum-sum subarray of a[0..n-1]
    static int[] kadane(int[] a, int n) {
        int best = a[0];
        int bl = 0;
        int br = 0;
        int cur = a[0];             // maximum sum of a subarray ending at i
        int cl = 0;
        int i = 1;
        while (i < n) {
            if (cur < 0) {
                cur = a[i];         // restart
                cl = i;
            } else {
                cur = cur + a[i];   // extend
            }
            if (cur > best) {
                best = cur;
                bl = cl;
                br = i;
            }
            i = i + 1;
        }
        return new int[]{bl, br, best};
    }

    // As above, but prints one row per iteration
    static int[] kadaneTrace(int[] a, int n) {
        int best = a[0], bl = 0, br = 0;
        int cur = a[0], cl = 0;
        System.out.printf("%3d %5d  %-24s %4d %3d %5d  (%d, %d)%n", 0, a[0], "initialization", cur, cl, best, bl, br);
        for (int i = 1; i < n; i++) {
            String action;
            if (cur < 0) {
                cur = a[i];
                cl = i;
                action = "restart (cur<0)";
            } else {
                cur = cur + a[i];
                action = "extend";
            }
            if (cur > best) {
                best = cur;
                bl = cl;
                br = i;
            }
            System.out.printf("%3d %5d  %-24s %4d %3d %5d  (%d, %d)%n", i, a[i], action, cur, cl, best, bl, br);
        }
        return new int[]{bl, br, best};
    }

    public static void main(String[] args) {
        int[] a = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
        int n = 16;
        System.out.println("a = " + Arrays.toString(a));
        System.out.printf("%3s %5s  %-24s %4s %3s %5s  (bl, br)%n", "i", "a[i]", "action", "cur", "cl", "best");
        int[] res = kadaneTrace(a, n);
        System.out.printf("result (%d, %d, %d)   (%d, %d, %d with indices from 1)%n", res[0], res[1], res[2], res[0] + 1, res[1] + 1, res[2]);
        int[] b = {-2, 5, -1, 3};
        int[] t = kadane(b, 4);
        System.out.printf("a = %s  Kadane(a, 4) = (%d, %d, %d)%n", Arrays.toString(b), t[0], t[1], t[2]);
    }
}

Writing Kadane.java


In [23]:
!javac Kadane.java && java Kadane

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
  i  a[i]  action                    cur  cl  best  (bl, br)
  0    13  initialization             13   0    13  (0, 0)
  1    -3  extend                     10   0    13  (0, 0)
  2   -25  extend                    -15   0    13  (0, 0)
  3    20  restart (cur<0)            20   3    20  (3, 3)
  4    -3  extend                     17   3    20  (3, 3)
  5   -16  extend                      1   3    20  (3, 3)
  6   -23  extend                    -22   3    20  (3, 3)
  7    18  restart (cur<0)            18   7    20  (3, 3)
  8    20  extend                     38   7    38  (7, 8)
  9    -7  extend                     31   7    38  (7, 8)
 10    12  extend                     43   7    43  (7, 10)
 11    -5  extend                     38   7    43  (7, 10)
 12   -22  extend                     16   7    43  (7, 10)
 13    15  extend                     31   7    43  (7, 10)
 14    -4  extend                    

**Try it yourself.** Change the array (for example with only negative numbers): Kadane restarts at every step and finds the largest element.

In [24]:
%%writefile ProvaKadane.java
import java.util.*;

public class ProvaKadane {
    public static void main(String[] args) {
        // Try an array of only negative numbers: Kadane restarts at every step and finds the largest element.
        int[] c = {-3, -1, -2, -5};
        int[] t = Kadane.kadane(c, c.length);
        System.out.println("(" + t[0] + ", " + t[1] + ", " + t[2] + ")");
    }
}

Writing ProvaKadane.java


In [25]:
!javac ProvaKadane.java && java ProvaKadane

(1, 1, -1)
